# Домашнее задание № 1

In [1]:
# !pip install razdel pymystem3 pymorphy3 pymorphy3-dicts-ru spacy
# !python -m spacy download ru_core_news_sm
import warnings
warnings.filterwarnings('ignore')

from razdel import sentenize, tokenize
from pymystem3 import Mystem
import pymorphy3
import spacy

mystem = Mystem()  # disambiguation=True по умолчанию
morph = pymorphy3.MorphAnalyzer()
nlp = spacy.load('ru_core_news_sm')

## Задание 1 (2 балла)

Найдите 1 любой способ сломать токенизацию на предложения функцией sentenize из библиотеки razdel. Придумайте (или найдите на каком-то корпусе) такое предложение (или несколько предложений), которое будет некорректно разобрано sentenize, но при этом будет грамматически корректным. 

In [2]:
def show(t):
    for i, s in enumerate(sentenize(t), 1):
        print(i, repr(s.text))
    print()

# Два грамматически корректных предложения, первое кончается сокращением «г.»
show("Это случилось в 1990 г. Потом всё изменилось.")
# Для сравнения: то же самое без сокращения — разбивается правильно
show("Это случилось в 1990 году. Потом всё изменилось.")
# Тот же эффект с другими сокращениями в конце предложения
show("Встреча назначена на 10 ч. Опаздывать нельзя.")
show("Мы обсудили вопросы экономики, политики и пр. Затем был обед.")

1 'Это случилось в 1990 г. Потом всё изменилось.'

1 'Это случилось в 1990 году.'
2 'Потом всё изменилось.'

1 'Встреча назначена на 10 ч. Опаздывать нельзя.'

1 'Мы обсудили вопросы экономики, политики и пр. Затем был обед.'



**Вывод.** `sentenize` считает, что после сокращения «г.» (а также «ч.», «пр.») предложение не заканчивается — обычно так и есть: «в 1990 г. в Москве…». Но если сокращение стоит в конце предложения, а следующее предложение начинается с заглавной буквы, точка одновременно закрывает и сокращение, и предложение (в русской орфографии вторую точку не ставят). razdel этого не видит и склеивает два предложения в одно. При этом с «т. д.» и «руб.» всё в порядке — для них, видимо, есть отдельные правила.

## 2. Токенизация Mystem vs razdel.tokenize (2 балла)


Токенизируйте текст с помощью razdel и с помощью Mystem. Найдите различия в токенизациях. Что по вашему работает лучше на приведенном тексте?

In [3]:
text = """
Вторым и третьим открытыми белыми карликами стали Сириус B и Процион B. В 1844 году директор Кёнигсбергской обсерватории Фридрих Бессель, анализируя данные наблюдений, которые велись с 1755 года, обнаружил, что Сириус, ярчайшая звезда земного неба, и Процион периодически, хотя и весьма слабо, отклоняются от прямолинейной траектории движения по небесной сфере[5]. Бессель пришёл к выводу, что у каждой из них должен быть близкий спутник. Сообщение было встречено скептически, поскольку слабый спутник оставался ненаблюдаемым, а его масса должна была быть достаточно велика — сравнимой с массой Сириуса и Проциона, соответственно.

В январе 1862 года Элвин Грэхэм Кларк, юстируя 18-дюймовый рефрактор, самый большой на то время телескоп в мире (Dearborn Telescope), впоследствии поставленный семейной фирмой Кларков в обсерваторию Чикагского университета, обнаружил в непосредственной близости от Сириуса тусклую звёздочку. Это был спутник Сириуса, Сириус B, предсказанный Бесселем[6]. А в 1896 году американский астроном Д. М. Шеберле открыл Процион B, подтвердив тем самым и второе предсказание Бесселя.

В 1915 году американский астроном Уолтер Сидней Адамс измерил спектр Сириуса B. Из измерений следовало, что его температура не ниже, чем у Сириуса A (по современным данным, температура поверхности Сириуса B составляет 25 000 K, а Сириуса A — 10 000 К), что, с учётом его в 10 000 раз меньшей, чем у Сириуса A, светимости указывает на очень малый радиус и, соответственно, высокую плотность — 106 г/см3 (плотность Сириуса ~0,25 г/см3, плотность Солнца ~1,4 г/см3).
"""

In [4]:
import difflib

razdel_tokens = [t.text for t in tokenize(text)]
mystem_tokens = [a['text'].strip() for a in mystem.analyze(text) if a['text'].strip()]
print('razdel:', len(razdel_tokens), 'токенов; mystem:', len(mystem_tokens), 'токенов\n')

sm = difflib.SequenceMatcher(a=razdel_tokens, b=mystem_tokens, autojunk=False)
for op, i1, i2, j1, j2 in sm.get_opcodes():
    if op != 'equal':
        print(f'razdel: {razdel_tokens[i1:i2]!s:30}  mystem: {mystem_tokens[j1:j2]}')

razdel: 292 токенов; mystem: 293 токенов

razdel: ['18-дюймовый']                 mystem: ['18', '-', 'дюймовый']
razdel: [')', ',']                      mystem: ['),']
razdel: [')', ',']                      mystem: ['),']
razdel: ['см', '3']                     mystem: ['см3']
razdel: ['0,25']                        mystem: ['0', ',', '25']
razdel: ['см', '3']                     mystem: ['см3']
razdel: ['1,4']                         mystem: ['1', ',', '4']
razdel: ['см', '3']                     mystem: ['см3']


**Различия:**

| Фрагмент | razdel | Mystem | Кто прав |
|---|---|---|---|
| `18-дюймовый` | один токен | `18` `-` `дюймовый` | razdel: это одно прилагательное |
| `0,25`, `1,4` | одно число | `0` `,` `25` | razdel: Mystem разрывает десятичную дробь, запятая становится «пунктуацией» |
| `г/см3` | `см` `3` | `см3` | спорно; Mystem сохраняет единицу «см³» целиком, razdel отрывает цифру (показатель степени потерян при копировании) |
| `),` | `)` `,` | `),` | razdel: у Mystem знаки препинания склеиваются в один небуквенный «токен» |

Кроме того, Mystem отдаёт пробелы и переводы строк как отдельные элементы (выше я их отфильтровал), а razdel возвращает только токены с позициями.

**Что лучше:** на этом тексте **razdel**. Здесь много чисел с дробной частью и составных слов через дефис, и razdel не разбивает их на части. Mystem прежде всего морфологический анализатор, а не токенизатор: он отделяет всё небуквенное от слов и ломает числа. Единственное, где Mystem выглядит естественнее, — `см3`.

# ваш код

## 3. Лемматизация Mystem vs Pymorphy (2 балла)

In [5]:
analysis = mystem.analyze(text)
rows = []
for a in analysis:
    if not a.get('analysis'):          # пробелы, пунктуация, числа, латиница
        continue
    word = a['text']
    lemma_mystem = a['analysis'][0]['lex']
    lemma_pymorphy = morph.parse(word)[0].normal_form   # токен взят из mystem
    rows.append((word, lemma_mystem, lemma_pymorphy))

print('Всего слов:', len(rows))
diff = [r for r in rows if r[1] != r[2].replace('ё', 'е')]
print('Различаются (без учёта ё):', len(diff), '\n')
print(f"{'слово':16}{'mystem':16}{'pymorphy'}")
for w, ml, pl in diff:
    print(f'{w:16}{ml:16}{pl}')

Всего слов: 199
Различаются (без учёта ё): 19 

слово           mystem          pymorphy
стали           становиться     стать
данные          данные          дать
обнаружил       обнаруживать    обнаружить
пришёл          приходить       прийти
встречено       встречать       встретить
его             его             он
Грэхэм          грэхэм          грэхэма
поставленный    поставлять      поставить
обнаружил       обнаруживать    обнаружить
предсказанный   предсказывать   предсказать
Д               д               далее
Шеберле         шеберль         шеберл
открыл          открывать       открыть
тем             то              тем
второе          второй          второе
измерил         измерять        измерить
его             его             он
ниже            низкий          ниже
меньшей         меньший         малый


**Значимые различия** (то, что Mystem пишет «е» вместо «ё», — «кенигсбергский», «учет», «звездочка» — я не считаю, это просто нормализация):

1. **Вид глагола.** На этих данных Mystem приводит совершенный вид к несовершенному: *обнаружил → обнаруживать*, *открыл → открывать*, *измерил → измерять*, *пришёл → приходить*, *встречено → встречать*, *поставленный → поставлять*, *предсказанный → предсказывать*, *стали → становиться*. pymorphy даёт правильный инфинитив того же вида: *обнаружить, открыть, измерить, прийти, встретить, поставить, предсказать, стать*. Здесь **лучше pymorphy**.
2. **Контекстная омонимия** — тут выигрывает дизамбигуация Mystem:
   * *данные* (существительное «данные наблюдений»): Mystem → *данные*, pymorphy → *дать* (причастие);
   * *второе* («второе предсказание»): Mystem → *второй*, pymorphy → *второе* (существительное, «блюдо»);
   * *тем* («тем самым»): Mystem → *то*, pymorphy → *тем* (наречие);
   * *его* (притяжательное): Mystem → *его*, pymorphy → *он*. Строго говоря, оба варианта допустимы.
3. **Инициалы и имена собственные:** *Д.* (инициал «Д. М. Шеберле»): pymorphy → *далее* (принял за сокращение), Mystem → *д*. *Грэхэм*: pymorphy → *грэхэма* (ошибка), Mystem → *грэхэм*. *Шеберле* (несклоняемая фамилия): оба ошиблись (*шеберль* / *шеберл*).
4. **Степени сравнения:** *ниже*: Mystem → *низкий* (правильно, это сравнительная степень), pymorphy → *ниже*. *меньшей*: Mystem → *меньший*, pymorphy → *малый*; оба варианта встречаются в разных стандартах разметки.

**Итог:** у pymorphy правильные глагольные леммы, но слов вне контекста он не различает. Mystem со снятой омонимией лучше справляется с «данными», «вторым», «тем», «ниже» и инициалами, но портит вид глаголов. Для этого научного текста существительных и терминов больше, чем глаголов, поэтому в целом я бы выбрала **Mystem**. Если важны глаголы, лучше pymorphy.

Важно: для пайморфи используйте токенизацию из mystem, чтобы исключить влияние токенизации на результат. Анализируйте только значимые различия, а не технические особенности (не сравнивайте скорость работы и удобность интерфейса).

В майстеме убедитесь, что используется дизамбигуация.

In [6]:
from spacy.tokens import Doc

# чтобы сравнение было честным, подаём в spaCy ту же токенизацию, что у mystem
items = [a for a in analysis if a['text'].strip()]
doc = nlp(Doc(nlp.vocab, words=[a['text'].strip() for a in items]))
spacy_lemmas = [(t.text, t.lemma_) for t, a in zip(doc, items) if a.get('analysis')]
assert [w for w, _ in spacy_lemmas] == [w for w, _, _ in rows]

print(f"{'слово':16}{'mystem':16}{'pymorphy':16}{'spacy'}")
for (w, ml, pl), (_, sl) in zip(rows, spacy_lemmas):
    if len({ml, pl.replace('ё', 'е'), sl.replace('ё', 'е')}) > 1:
        print(f'{w:16}{ml:16}{pl:16}{sl}')

слово           mystem          pymorphy        spacy
стали           становиться     стать           стать
данные          данные          дать            данные
которые         который         который         которые
обнаружил       обнаруживать    обнаружить      обнаружить
пришёл          приходить       прийти          прийти
них             они             они             них
было            быть            быть            было
встречено       встречать       встретить       встретить
его             его             он              он
Проциона        процион         процион         проциона
Грэхэм          грэхэм          грэхэма         грэхэм
то              то              то              тот
поставленный    поставлять      поставить       поставить
Кларков         кларк           кларк           кларков
Чикагского      чикагский       чикагский       чикагского
обнаружил       обнаруживать    обнаружить      обнаружить
предсказанный   предсказывать   предсказать     предсказа

In [7]:
import collections
agree = collections.Counter()
for (w, ml, pl), (_, sl) in zip(rows, spacy_lemmas):
    ml, pl, sl = ml, pl.replace('ё', 'е'), sl.replace('ё', 'е')
    agree['spacy == mystem'] += sl == ml
    agree['spacy == pymorphy'] += sl == pl
    agree['mystem == pymorphy'] += ml == pl
print(len(rows), 'слов'); print(dict(agree))

199 слов
{'spacy == mystem': 172, 'spacy == pymorphy': 180, 'mystem == pymorphy': 180}


**Наблюдения.** Лемматизатор spaCy для русского языка внутри использует pymorphy3, но выбирает разбор по частеречному тегу из своей нейросетевой модели. Токенизация одинаковая (подали токены Mystem), поэтому сравнение честное.

* С pymorphy spaCy совпадает в 180 случаях из 199, с Mystem — в 172. Глаголы у него, как и у pymorphy, правильного вида: *обнаружить, открыть, измерить, прийти, поставить*.
* Благодаря тегам исправлена часть контекстных ошибок pymorphy: *второе → второй*, *ниже → низкий*, *данные → данные* (а не *дать*), *на то время → тот*, *Грэхэм → грэхэм*, *Д. → д*, *Шеберле → шеберле* (единственная правильная лемма из трёх).
* Зато появились новые ошибки, в основном когда модель ставит неверный тег (часто PROPN), и тогда лемма остаётся словоформой: *Проциона → проциона*, *Бесселя → бесселя*, *Бесселем → бесселем*, *Кларков → кларков*, *Чикагского → чикагского*, *данным → данным*, *было → было*, *них → них*, *которые → которые*, *самым → самым*, *Из → Из* (лемма даже не приведена к нижнему регистру).

Итого spaCy где-то посередине: глаголы и часть омонимии у него лучше, чем у Mystem и pymorphy, но маленькая модель `ru_core_news_sm` часто вообще не лемматизирует имена собственные и местоимения. На этом тексте Mystem надёжнее.

# ваш код

## 4. Лемматизация в SpaCy (2 балла)

С помощью Spacy (модель для русского языка) лемматизируйте тот же текст. Проверьте есть ли различия с Mystem и Pymoprhy.

In [8]:
from pathlib import Path
import re

text = Path("anna_karenina.txt").read_text(encoding="utf-8")

print("Количество символов:", len(text))
print(text[:500])

Количество символов: 1710408
Annotation


«Анна Каренина» – это сложное, психологически утонченное, остропроблемное произведение, насыщенное приметами времени. Л.Н. Толстой на страницах произведения показывает, как рушатся остатки патриархального уклада жизни в России под натиском буржуазного прогресса, как падают нравы, ослабевают семейные устои, вырождается аристократия.

Роман во многом автобиографичен. Работая над ним, Толстой уяснял взгляд на современность и свою собственную жизнь.





* * *



Лев Николаевич Толстой



In [9]:
paragraphs = [
    p.strip()
    for p in re.split(r"\n\s*\n", text)
    if p.strip()
]

print("Количество абзацев:", len(paragraphs))

Количество абзацев: 7821


Разработайте поиск по неточному описанию эпизода художественного произведения. Сравните поиск по словам, по леммам и один дополнительный способ с использованием spaCy. Найдите релевантный фрагмент и выведите окружающий контекст. Объясните, какие методы помогли и почему.

In [10]:
query = "Левин встречается с Кити на катке и сам катается на коньках."

### Подход

* **Единица поиска** — окно из 3 подряд идущих абзацев: эпизод в романе обычно занимает несколько коротких абзацев с репликами, и слова из запроса распределены по ним.
* **Ранжирование** — BM25 (реализован ниже вручную). Он учитывает редкость слова (IDF) и насыщается по частоте, поэтому частые «Левин»/«Кити» не перевешивают редкие «каток»/«коньки».
* Сравниваются три представления текста и запроса:
  1. **слова**: словоформы в нижнем регистре (токенизация razdel);
  2. **леммы**: Mystem;
  3. **spaCy**: леммы spaCy, **только знаменательные части речи** (NOUN, PROPN, VERB, ADJ), без стоп-слов + **фильтр по персонажам**: в окне должны упоминаться все персонажи, которых NER spaCy нашёл в запросе (`PER`). Имена используются только для фильтрации, а ранжирование идёт по остальным словам («что происходит»).

In [11]:
import math
from collections import Counter

STOP = {'и', 'в', 'с', 'на', 'сам', 'по', 'к', 'а', 'не', 'что', 'он', 'она', 'быть'}

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = [Counter(d) for d in docs]
        self.lens = [len(d) for d in docs]
        self.avg = sum(self.lens) / len(self.lens)
        df = Counter(w for d in self.docs for w in d)
        n = len(docs)
        self.idf = {w: math.log(1 + (n - f + 0.5) / (f + 0.5)) for w, f in df.items()}
        self.k1, self.b = k1, b

    def score(self, query, i):
        d, L, s = self.docs[i], self.lens[i], 0.0
        for w in set(query):
            tf = d.get(w, 0)
            if tf:
                s += self.idf[w] * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * L / self.avg))
        return s

    def search(self, query, top=5, allowed=None):
        idx = range(len(self.docs)) if allowed is None else allowed
        return sorted(((self.score(query, i), i) for i in idx), reverse=True)[:top]

WIN = 3
def windows(par_tokens):
    """Окно = абзацы i..i+WIN-1, токены склеиваются."""
    return [sum(par_tokens[i:i + WIN], []) for i in range(len(par_tokens) - WIN + 1)]

def show_hits(hits, label, query_terms, ctx=False):
    print(f'=== {label} | запрос: {query_terms}')
    for s, i in hits:
        print(f'  score={s:5.2f}  абзацы {i}-{i + WIN - 1}: {paragraphs[i][:110]!r}')
    if ctx:
        s, i = hits[0]
        print('\n--- контекст лучшего результата (±1 абзац) ---')
        for j in range(max(0, i - 1), min(len(paragraphs), i + WIN + 1)):
            print(f'[{j}]', paragraphs[j], '\n')
    print()

GOLD = range(285, 335)  # где на самом деле сцена на катке (часть 1, гл. 9): для оценки
def rank_of_gold(bm, q, allowed=None):
    ranked = bm.search(q, top=len(bm.docs), allowed=allowed)
    for r, (s, i) in enumerate(ranked, 1):
        if i in GOLD and s > 0:
            return r

#### 1. Поиск по словам

In [12]:
word_pars = [[t.text.lower() for t in tokenize(p) if t.text.isalpha()] for p in paragraphs]
bm_words = BM25(windows(word_pars))

q_words = [t.text.lower() for t in tokenize(query) if t.text.isalpha() and t.text.lower() not in STOP]
hits = bm_words.search(q_words)
show_hits(hits, 'СЛОВА', q_words)
print('Первое окно из настоящей сцены на катке на месте:', rank_of_gold(bm_words, q_words))

=== СЛОВА | запрос: ['левин', 'встречается', 'кити', 'катке', 'катается', 'коньках']
  score=19.32  абзацы 218-220: '– Ах, об этом тоже поговорим после, – опять до ушей покраснев, сказал Левин.'
  score=18.64  абзацы 217-219: '– Да ничего, – отвечал Облонский. – Мы поговорим. Да ты зачем, собственно, приехал?'
  score=17.37  абзацы 219-221: '– Ну, хорошо. Понятно, – сказал Степан Аркадьич. – Так видишь ли: – я бы позвал тебя к себе, но жена не совсем'
  score=11.18  абзацы 328-330: 'В это время один из молодых людей, лучший из новых конькобежцев, с папироской во рту, в коньках, вышел из кофе'
  score=10.88  абзацы 327-329: '«Боже мой, что я сделал! Господи боже мой! помоги мне, научи меня», – говорил Левин, молясь и вместе с тем чув'

Первое окно из настоящей сцены на катке на месте: 4


#### 2. Поиск по леммам (Mystem)

In [13]:
%%time
# прогоняем весь роман через mystem одним вызовом: абзацы разделены маркером-строкой
SEP = 'qqqsepqqq'
lem = mystem.lemmatize(f' {SEP} '.join(paragraphs))
lemma_pars, cur = [], []
for l in lem:
    l = l.strip()
    if l == SEP:
        lemma_pars.append(cur); cur = []
    elif l.isalpha():
        cur.append(l)
lemma_pars.append(cur)
assert len(lemma_pars) == len(paragraphs)

CPU times: user 1min 21s, sys: 6.7 s, total: 1min 28s
Wall time: 1min 34s


In [14]:
bm_lemmas = BM25(windows(lemma_pars))
q_lemmas = [l.strip() for l in mystem.lemmatize(query) if l.strip().isalpha() and l.strip() not in STOP]
hits = bm_lemmas.search(q_lemmas)
show_hits(hits, 'ЛЕММЫ', q_lemmas, ctx=True)
print('Первое окно из настоящей сцены на катке на месте:', rank_of_gold(bm_lemmas, q_lemmas))

=== ЛЕММЫ | запрос: ['левин', 'встречаться', 'кити', 'каток', 'кататься', 'конек']
  score=19.77  абзацы 289-291: 'Он шел по дорожке к катку и говорил себе: – «Надо не волноваться, надо успокоиться. О чем ты? Чего ты? Молчи, '
  score=19.66  абзацы 300-302: '– Да, я когда-то со страстью катался; мне хотелось дойти до совершенства.'
  score=19.57  абзацы 301-303: '– Вы все, кажется, делаете со страстью, – сказала она улыбаясь. – Мне так хочется посмотреть, как вы катаетесь'
  score=19.31  абзацы 291-293: 'На льду собирались в этот день недели и в эту пору дня люди одного кружка, все знакомые между собою. Были тут '
  score=18.96  абзацы 290-292: 'Он узнал, что она тут, по радости и страху, охватившим его сердце. Она стояла, разговаривая с дамой, на против'

--- контекст лучшего результата (±1 абзац) ---
[288] Был ясный морозный день. У подъезда рядами стояли кареты, сани, ваньки, жандармы. Чистый народ, блестя на ярком солнце шляпами, кишел у входа и по расчищенным дорожкам, между русск

#### 3. spaCy: леммы знаменательных слов + фильтр по персонажам (NER)

In [15]:
%%time
CONTENT = {'NOUN', 'PROPN', 'VERB', 'ADJ'}
spacy_pars, persons_pars = [], []
for d in nlp.pipe(paragraphs, batch_size=256, disable=['parser']):
    spacy_pars.append([t.lemma_.lower() for t in d if t.pos_ in CONTENT and not t.is_stop and t.is_alpha])
    persons_pars.append({t.lemma_.lower() for e in d.ents if e.label_ == 'PER' for t in e})

CPU times: user 1min 3s, sys: 1.99 s, total: 1min 5s
Wall time: 1min 5s


In [16]:
bm_spacy = BM25(windows(spacy_pars))

qdoc = nlp(query)
q_spacy = [t.lemma_.lower() for t in qdoc if t.pos_ in CONTENT and not t.is_stop and t.is_alpha]
q_persons = {t.lemma_.lower() for e in qdoc.ents if e.label_ == 'PER' for t in e}
print('Персонажи в запросе (NER):', q_persons)

# окна, где упомянуты все персонажи из запроса (по NER или просто по лемме)
allowed = [i for i in range(len(bm_spacy.docs))
           if q_persons <= set().union(*persons_pars[i:i + WIN], *map(set, spacy_pars[i:i + WIN]))]
print('Окон после фильтра по персонажам:', len(allowed), 'из', len(bm_spacy.docs), '\n')

# персонажи уже учтены фильтром, ранжируем только по остальным (содержательным) словам
q_action = [w for w in q_spacy if w not in q_persons]
hits = bm_spacy.search(q_action, allowed=allowed)
show_hits(hits, 'SPACY', q_action, ctx=True)
print('Первое окно из настоящей сцены на катке на месте:', rank_of_gold(bm_spacy, q_action, allowed))

Персонажи в запросе (NER): {'кити', 'левин'}
Окон после фильтра по персонажам: 547 из 7819 

=== SPACY | запрос: ['встречаться', 'каток', 'кататься', 'конёк']
  score=14.73  абзацы 290-292: 'Он узнал, что она тут, по радости и страху, охватившим его сердце. Она стояла, разговаривая с дамой, на против'
  score=13.70  абзацы 218-220: '– Ах, об этом тоже поговорим после, – опять до ушей покраснев, сказал Левин.'
  score=13.37  абзацы 217-219: '– Да ничего, – отвечал Облонский. – Мы поговорим. Да ты зачем, собственно, приехал?'
  score=12.91  абзацы 531-533: '– Ах, как же! Я все записываю. Ну что, Кити, ты опять каталась на коньках?..'
  score=11.79  абзацы 291-293: 'На льду собирались в этот день недели и в эту пору дня люди одного кружка, все знакомые между собою. Были тут '

--- контекст лучшего результата (±1 абзац) ---
[289] Он шел по дорожке к катку и говорил себе: – «Надо не волноваться, надо успокоиться. О чем ты? Чего ты? Молчи, глупое», – обращался он к своему сердцу. И чем больш

#### Ещё пара запросов для проверки

In [17]:
def compare(q):
    qw = [t.text.lower() for t in tokenize(q) if t.text.isalpha() and t.text.lower() not in STOP]
    ql = [l.strip() for l in mystem.lemmatize(q) if l.strip().isalpha() and l.strip() not in STOP]
    qd = nlp(q)
    qs = [t.lemma_.lower() for t in qd if t.pos_ in CONTENT and not t.is_stop and t.is_alpha]
    qp = {t.lemma_.lower() for e in qd.ents if e.label_ == 'PER' for t in e}
    al = [i for i in range(len(bm_spacy.docs))
          if qp <= set().union(*persons_pars[i:i + WIN], *map(set, spacy_pars[i:i + WIN]))]
    print('ЗАПРОС:', q)
    qs = [w for w in qs if w not in qp]
    for name, bm, qq, a in [('слова', bm_words, qw, None), ('леммы', bm_lemmas, ql, None), ('spacy', bm_spacy, qs, al)]:
        s, i = bm.search(qq, top=1, allowed=a)[0]
        print(f'  {name:6} абзацы {i}-{i+WIN-1}: {paragraphs[i][:100]!r}')
    print()

compare("Анна бросается под поезд на станции.")
compare("Вронский падает с лошади на скачках, и лошадь ломает спину.")
compare("Левин косит траву вместе с мужиками.")

ЗАПРОС: Анна бросается под поезд на станции.
  слова  абзацы 7423-7425: 'Простившись с княгиней, Сергей Иваныч вместе с подошедшим Катавасовым вошел в битком набитый вагон, '
  леммы  абзацы 701-703: '– Пойдемте, теперь мало народа, – сказал Вронский.'
  spacy  абзацы 7319-7321: '«Да, надо ехать на станцию железной дороги, а если нет, то поехать туда и уличить его». Анна посмотр'

ЗАПРОС: Вронский падает с лошади на скачках, и лошадь ломает спину.
  слова  абзацы 2053-2055: '«О, прелесть моя!» – думал он на Фру-Фру, прислушиваясь к тому, что происходило сзади. «Перескочил!»'
  леммы  абзацы 2042-2044: 'В первые минуты Вронский еще не владел ни собою, ни лошадью. Он до первого препятствия, реки, не мог'
  spacy  абзацы 2042-2044: 'В первые минуты Вронский еще не владел ни собою, ни лошадью. Он до первого препятствия, реки, не мог'

ЗАПРОС: Левин косит траву вместе с мужиками.
  слова  абзацы 2623-2625: 'Как ни было легко косить мокрую и слабую траву, но трудно было спускаться и поднимат

### Выводы

| Способ | На каком месте настоящая сцена на катке |
|---|---|
| слова | 4 (первые три окна — разговор Левина с Облонским, где каток только упоминается) |
| леммы Mystem | **1** |
| spaCy (знаменательные леммы + фильтр по персонажам) | **1** |

* **По словам.** Запрос в одних формах («на катке», «катается», «на коньках»), а в романе в других («к катку», «конца катка», «коньки», «катавшихся»). Совпадают в основном имена и одна случайная форма «коньках», поэтому вверх выходит абзац 219, где Облонский мельком говорит Левину, что Кити бывает на катке.
* **По леммам (Mystem).** *катке/катку/катка → каток*, *коньках/коньками → конек*, *катается/катавшихся → кататься*. После лемматизации запрос и текст совпадают по содержательным словам, и все пять лучших окон — из сцены в Зоологическом саду (часть 1, гл. 9). Контекст выше: Левин идёт к катку и узнаёт Кити среди катающихся.
* **spaCy.** Части речи отсекают служебные слова, NER выделяет в запросе персонажей. Персонажей используем как фильтр (из 7819 окон остаётся 547, где есть и Левин, и Кити), а ранжируем по словам о том, *что происходит*. Тогда частые имена не забивают рейтинг, и сцена тоже оказывается на 1-м месте. В первом варианте, где имена оставались в BM25 (я проверяла), сцена была только на 3-м месте.
* **Дополнительные запросы.** Скачки и покос нашли и леммы, и spaCy. **Гибель Анны** («бросается под поезд на станции») не нашёл ни один способ: у Толстого «упасть под… вагон», «рельсы», а слова «поезд» и «бросаться» почти не употребляются. spaCy с фильтром «Анна» выдал абзацы 7319–7321 (Анна решает ехать на станцию). Это та же глава, за ~35 абзацев до нужного места (абзац 7355), но не сама сцена.

**Что помогло и почему:** главное — **лемматизация**, потому что описание эпизода почти всегда написано другими словоформами, чем оригинал. **Окно из нескольких абзацев + BM25** поднимает окна, где редкие слова запроса встречаются *вместе*. **spaCy (части речи + NER)** убирает шум и отделяет «кто» от «что происходит». Чего этим методам не хватает — синонимии («поезд» ≠ «вагон», «бросается» ≠ «упасть»). Для этого нужны векторные представления (`ru_core_news_lg` или sentence-эмбеддинги).